# Create Epilepsy Research UK Awards

178 funded projects via the epilepsyresearch.org.uk WP REST `research_portfolio` endpoint. Title-only ingest — PI/institution/amount are only on Elementor detail pages with no clean fields, so lead_investigator is NULL (§6.7 waiver). Title is the linkage signal.

provenance `eruk`, priority 335. F4320320012 (Path A).

Source parquet: s3://openalex-ingest/awards/eruk/eruk_projects.parquet
Built by scripts/local/eruk_to_s3.py

## Step 1: Create Staging Table from S3

In [ ]:
%sql
CREATE OR REPLACE TABLE openalex.awards.eruk_raw
USING delta
AS
SELECT *, current_timestamp() as databricks_ingested_at
FROM parquet.`s3a://openalex-ingest/awards/eruk/eruk_projects.parquet`;


In [ ]:
%sql
SELECT COUNT(*) as total FROM openalex.awards.eruk_raw;

In [ ]:
%sql
DESCRIBE openalex.awards.eruk_raw;

In [ ]:
%sql
SELECT * FROM openalex.awards.eruk_raw LIMIT 5;

## Step 2: Create ERUK Awards Table

In [ ]:
%sql
CREATE OR REPLACE TABLE openalex.awards.eruk_awards
USING delta
AS
WITH
the_funder AS (
    SELECT funder_id, display_name, ror_id, doi
    FROM openalex.common.funder
    WHERE funder_id = 4320320012  -- Epilepsy Research UK
),
awards_transformed AS (
    SELECT
        abs(xxhash64(CONCAT(f.funder_id, ':', LOWER(g.funder_award_id)))) % 9000000000 as id,
        g.title as display_name,
        CAST(NULL AS STRING) as description,
        f.funder_id,
        g.funder_award_id as funder_award_id,
        CAST(NULL AS DECIMAL(18,2)) as amount,
        CAST(NULL AS STRING) as currency,
        struct(
            CONCAT('https://openalex.org/F', f.funder_id) as id,
            f.display_name, f.ror_id, f.doi
        ) as funder,
        'research' as funding_type,
        CAST(NULL AS STRING) as funder_scheme,
        'eruk' as provenance,
        CAST(NULL AS DATE) as start_date,
        CAST(NULL AS DATE) as end_date,
        CAST(NULL AS INT) as start_year,
        CAST(NULL AS INT) as end_year,
        CASE
            WHEN CAST(NULL AS STRING) IS NOT NULL THEN
                struct(
                    CAST(NULL AS STRING) as given_name,
                    CAST(NULL AS STRING) as family_name,
                    CAST(NULL AS STRING) as orcid,
                    CAST(NULL AS DATE) as role_start,
                    struct(
                        CAST(NULL AS STRING) as name,
                        CAST(NULL AS STRING) as country,
                        CAST(NULL AS ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>) as ids
                    ) as affiliation
                )
            ELSE NULL
        END as lead_investigator,
        CAST(NULL AS STRUCT<
            given_name:STRING, family_name:STRING, orcid:STRING, role_start:DATE,
            affiliation:STRUCT<name:STRING, country:STRING, ids:ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>>
        >) as co_lead_investigator,
        CAST(NULL AS ARRAY<STRUCT<
            given_name:STRING, family_name:STRING, orcid:STRING, role_start:DATE,
            affiliation:STRUCT<name:STRING, country:STRING, ids:ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>>
        >>) as investigators,
        g.landing_page_url as landing_page_url,
        CAST(NULL AS STRING) as doi,
        CAST(NULL AS STRING) as works_api_url,
        current_timestamp() as created_date,
        current_timestamp() as updated_date
    FROM openalex.awards.eruk_raw g
    CROSS JOIN the_funder f
)
SELECT * FROM awards_transformed;


## Step 3: Insert into openalex_awards_raw

In [ ]:
%sql
DELETE FROM openalex.awards.openalex_awards_raw
WHERE provenance = 'eruk' AND priority = 335;

INSERT INTO openalex.awards.openalex_awards_raw
SELECT
    id, display_name, description, funder_id, funder_award_id, amount, currency,
    funder, funding_type, funder_scheme, provenance, start_date, end_date,
    start_year, end_year, lead_investigator, co_lead_investigator, investigators,
    landing_page_url, doi, works_api_url, created_date, updated_date,
    335 as priority
FROM openalex.awards.eruk_awards;

## Verification Queries

In [ ]:
%sql
SELECT COUNT(*) as total_awards FROM openalex.awards.eruk_awards;

In [ ]:
%sql
SELECT id, display_name, funder_award_id, amount, currency, start_year FROM openalex.awards.eruk_awards LIMIT 10;

In [ ]:
%sql
SELECT funder.display_name, COUNT(*) cnt FROM openalex.awards.eruk_awards GROUP BY 1 ORDER BY cnt DESC;

In [ ]:
%sql
SELECT COUNT(*) total, COUNT(display_name) has_title, COUNT(amount) has_amount, COUNT(lead_investigator) has_pi,
  ROUND(try_divide(COUNT(amount),COUNT(*))*100,1) pct_amount FROM openalex.awards.eruk_awards;

In [ ]:
%sql
-- Step 6.4a PI frequency
SELECT lead_investigator.given_name given, lead_investigator.family_name family, COUNT(*) n
FROM openalex.awards.eruk_awards GROUP BY 1,2 ORDER BY n DESC LIMIT 20;

In [ ]:
%sql
SELECT lead_investigator.affiliation.name institution, COUNT(*) cnt FROM openalex.awards.eruk_awards
WHERE lead_investigator.affiliation.name IS NOT NULL GROUP BY 1 ORDER BY cnt DESC LIMIT 20;

In [ ]:
%sql
SELECT COUNT(*) as eruk_in_raw FROM openalex.awards.openalex_awards_raw WHERE provenance='eruk' AND priority=335;